In [2]:
import re
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [3]:
def clean_text(text: str) -> str:
    # Remove timestamps from web-printed PDFs (Aetna)
    text = re.sub(r'\d+/\d+/\d+,\s+\d+:\d+\s+[AP]M', '', text)
    # Remove page fraction artifacts like "3/333"
    text = re.sub(r'\n\d+/\d+\n', '\n', text)
    # Remove repeated site headers (Aetna)
    text = re.sub(r'Genetic Testing - Medical Clinical Policy Bulletins \| Aetna', '', text)
    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # Remove Medi-Cal change mark symbols
    text = text.replace('‹‹', '').replace('››', '')
    # Collapse excessive whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

print("Clean function defined.")

Clean function defined.


### Cleaning a single file using regex

In [4]:
DATA_DIR = Path("../data")
VECTORSTORE_DIR = Path("../vectorstore/chroma_db")

# Example: load a single document to demonstrate the pipeline
sample_file = DATA_DIR / "medi_cal_molecular_pathology.pdf"

loader = PyPDFLoader(str(sample_file))
sample_docs = loader.load()

print(f"Loaded: {sample_file.name}")
print(f"Pages: {len(sample_docs)}")
print(f"\nSample text (first 300 chars):")
print(sample_docs[0].page_content[:300])

Loaded: medi_cal_molecular_pathology.pdf
Pages: 124

Sample text (first 300 chars):
path molec  
1 
Part 2 – Pathology: Molecular Pathology  Pathology: Molecular Pathology  
Page updated: September 2024  
This section contains information to help providers bill for clinical laboratory tests or 
examinations related to molecular pathology and diagnostic services.  
Molecular Patholo


In [5]:
# Apply cleaning to sample docs
for doc in sample_docs:
    doc.page_content = clean_text(doc.page_content)

print("Cleaning applied.")
print(f"\nSample text after cleaning (first 300 chars):")
print(sample_docs[0].page_content[:300])

Cleaning applied.

Sample text after cleaning (first 300 chars):
path molec 
1 
Part 2 – Pathology: Molecular Pathology Pathology: Molecular Pathology 
Page updated: September 2024 
This section contains information to help providers bill for clinical laboratory tests or 
examinations related to molecular pathology and diagnostic services. 
Molecular Pathology Co


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(sample_docs)

print(f"Pages: {len(sample_docs)}")
print(f"Chunks: {len(chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print(f"\nSample chunk:")
print(chunks[10].page_content)

Pages: 124
Chunks: 408
Avg chunk size: 410 chars

Sample chunk:
81161 
DMD (dystrophin) 
deletion analysis, and 
duplication analysis, if 
performed No ICD-10-CM diagnosis code G71.0 
(muscular dystrophy) is required on 
the claim. Once -in-a-
lifetime


## Full Pipeline — All Documents

The cells above demonstrate the pipeline on a single file.  
The cells below run the full corpus: 5 documents across Medi-Cal, Blue Shield CA, and Aetna.


In [7]:
pdf_loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

txt_loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True
)

pdf_docs = pdf_loader.load()
txt_docs = txt_loader.load()
all_documents = pdf_docs + txt_docs

print(f"Loaded {len(all_documents)} pages from "
      f"{len(set(d.metadata['source'] for d in all_documents))} documents")

100%|██████████| 2/2 [00:00<00:00, 1662.43it/s]

Loaded 565 pages from 7 documents


In [8]:
# Apply cleaning to all documents
for doc in all_documents:
    doc.page_content = clean_text(doc.page_content)

# Filter out non-clinical files
all_documents = [
    doc for doc in all_documents
    if "sources.txt" not in doc.metadata["source"]
]

print(f"Documents after filtering: "
      f"{len(set(d.metadata['source'] for d in all_documents))}")
print(f"Pages after filtering: {len(all_documents)}")
print("\nSources in corpus:")
for source in sorted(set(d.metadata['source'] for d in all_documents)):
    print(f"  {Path(source).name}")

Documents after filtering: 6
Pages after filtering: 564

Sources in corpus:
  aetna_genetic_testing_cpb0140.pdf
  blue_shield_ca_genetic_testing_dev_delay.pdf
  cms_prior_auth_overview.txt
  medi_cal_genetic_counseling.pdf
  medi_cal_mckt_provider_training.pdf
  medi_cal_molecular_pathology.pdf


In [9]:
all_chunks = splitter.split_documents(all_documents)

print(f"Total pages: {len(all_documents)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in all_chunks) // len(all_chunks)} chars")
print(f"\nChunks by document:")
from collections import Counter
source_counts = Counter(Path(c.metadata['source']).name for c in all_chunks)
for source, count in sorted(source_counts.items()):
    print(f"  {source}: {count} chunks")

Total pages: 564
Total chunks: 2511
Avg chunk size: 417 chars

Chunks by document:
  aetna_genetic_testing_cpb0140.pdf: 1605 chunks
  blue_shield_ca_genetic_testing_dev_delay.pdf: 268 chunks
  cms_prior_auth_overview.txt: 12 chunks
  medi_cal_genetic_counseling.pdf: 47 chunks
  medi_cal_mckt_provider_training.pdf: 171 chunks
  medi_cal_molecular_pathology.pdf: 408 chunks


## Retrieval Strategy Note

Aetna CPB 0140 accounts for ~64% of corpus chunks (1605/2511) due to its 
comprehensive 333-page structure. To prevent Aetna from dominating retrieval 
on all queries, we use metadata filtering at query time to target specific 
payers when needed.

This is implemented in the RAG chain notebook.

In [11]:
print("Loading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model ready.")

Loading embedding model...


/Users/prasunamannava/clinical-prior-auth-assistant/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/prasunamannava/clinical-prior-auth-assistant/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding model ready.


In [12]:
import shutil

if VECTORSTORE_DIR.exists():
    shutil.rmtree(VECTORSTORE_DIR)
    print("Vectorstore cleared.")

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=str(VECTORSTORE_DIR)
)

print(f"Stored {vectorstore._collection.count()} chunks in Chroma.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Stored 2511 chunks in Chroma.
